In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
da_to_de_dir = "/data7/deepro/starrseq/papers/results/6_link_da_enhancers_to_de_genes/data/da_enhancers_to_de_genes_links"

In [3]:
libraries = ["ATF2", "CTCF", "FOXA1", "LEF1", "SCRT1", "TCF7L2", "16P12_1"]
rtypes = ["induced", "gained", "repressed", "lost"]
priority = ["gained", "lost", "induced", "repressed"]
cat_type = pd.api.types.CategoricalDtype(priority, ordered=True)

In [4]:
act_exp_dfs = []
for lib in libraries:
    lib_dfs = []
    for rtype in rtypes:
        filename = os.path.join(da_to_de_dir, lib, f"{rtype}_merged_consistent.csv")
        if not os.path.exists(filename):
            continue
        df = pd.read_csv(filename, index_col="chrom_coord")
        df["category"] = rtype
        df[f"{lib}_neglog10padj"] = -df[f"{lib}_padj"].apply(lambda x: np.log10(x) if x > 0 else pd.NA)
        relevant_columns = ["category", f"{lib}_peak", f"{lib}_log2FoldChange", f"{lib}_neglog10padj", "gene_id", "gene_name", "log2FoldChange", "neglog10padj"]
        pval_threshold = -np.log10(0.05)
        df = df.loc[(df[f"{lib}_neglog10padj"] > pval_threshold)&(df["neglog10padj"] > pval_threshold), relevant_columns]
        lib_dfs.append(df)

    lib_df = pd.concat(lib_dfs)
    lib_df["category"] = lib_df["category"].astype(cat_type)
    # one row per chrom_coord
    lib_df = lib_df.sort_values(["category", f"{lib}_neglog10padj"], ascending=[True, False]).groupby(level=0, as_index=True).head(1)

    column_rename_dict = {
        "category": (lib, "activity", "category"),
        f"{lib}_peak": (lib, "activity", "peak"),
        f"{lib}_log2FoldChange": (lib, "activity", "log2FoldChange"),
        f"{lib}_neglog10padj": (lib, "activity", "neglog10padj"),
        "gene_id": (lib, "expression", "gene_id"),
        "gene_name": (lib, "expression", "gene_name"),
        "log2FoldChange": (lib, "expression", "log2FoldChange"),
        "neglog10padj": (lib, "expression", "neglog10padj")
    }
    lib_df.columns = pd.MultiIndex.from_tuples([column_rename_dict[col] for col in lib_df.columns])
    
    act_exp_dfs.append(lib_df)

act_exp_df = pd.concat(act_exp_dfs, axis=1)

In [5]:
savefile = "/data7/deepro/starrseq/papers/results/6_link_da_enhancers_to_de_genes/data/tables/supplementary_data3.xlsx"
act_exp_df.to_excel(savefile)
